In [3]:
import getpass
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

engine = create_engine(
    URL.create(
        "postgresql+pg8000",
        username="postgres",
        password=getpass.getpass("PostgreSQL password: "),
        host="localhost",
        port=5432,
        database="retail_db",
    )
)

df = pd.read_sql("SELECT * FROM online_retail_raw", engine)
df.shape

(1067371, 8)

In [4]:
df.info()
df.isnull().sum()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   invoice      1067371 non-null  object 
 1   stockcode    1067371 non-null  object 
 2   description  1062989 non-null  object 
 3   quantity     1067371 non-null  int64  
 4   invoicedate  1067371 non-null  object 
 5   price        1067371 non-null  float64
 6   customer_id  824364 non-null   float64
 7   country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


,quantity,price,customer_id
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [5]:
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
df["is_guest"] = df["customer_id"].isnull()

before = len(df)
clean = df[(df["quantity"] > 0) & (df["price"] > 0) & (~df["invoice"].str.startswith("C"))]
dropped = before - len(clean)
print(f"Dropped {dropped:,} rows ({100*dropped/before:.2f}%)")

Dropped 25,701 rows (2.41%)


## Data Quality Summary

- Total rows: 1,067,371
- Rows with no customer_id: 243,007 (22.77% of rows, 15.40% of revenue). Kept and flagged with `is_guest`; they cannot be linked to a customer, so they are excluded only from customer-level RFM.
- Dropped rows (cancelled invoices, quantity <= 0, price <= 0): 25,701 (2.41%), within the 5% limit.

In [6]:
clean.to_csv("../data/interim/online_retail_clean.csv.gz", index=False)